# Composing your own module

## Setting up DSPy

### Load environment variables

In [1]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [2]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [3]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## Composing modules to build an ensemble

We will composed our own `HaikuEnsemble`.

LLM-written poems are a roll of the dice. Sometimes their haikus are evocative; other times they’re predictable and bland. To increase our program’s odds of success, we’re going to roll the dice several times, then select the best candidate.

![](../assets/haiku_ensemble_module.png)

Here’s what this looks like in code:

In [4]:
class HaikuEnsemble(dspy.Module):
    def __init__(self, n: int = 3):
        super().__init__()
        self.n = n  
        # Module 1 generates several haikus
        self.writer = dspy.ChainOfThought(
            "location, season, mood, num_haikus: int -> haikus: list[str]",
        )
        # Module 2 picks the most evocative
        self.judge = dspy.ChainOfThought(
            "location, season, mood, candidates: list[str] -> most_evocative_index: int"
        )

    def forward(self, location: str, season: str, mood: str) -> dspy.Prediction:
        candidates = self.writer(
            location=location, season=season, mood=mood, num_haikus=self.n,
        ).haikus
        verdict = self.judge( 
            location=location, season=season, mood=mood, candidates=candidates,
        )
        return dspy.Prediction(
            haiku=candidates[verdict.most_evocative_index],
            candidates=candidates,
            reasoning=verdict.reasoning,
        )

When constructing a module, we need to write two functions:

1. `__init__` sets up our initial state and defines our submodules.
2. `forward` handles what happens when we call our program, accepting inputs and shepherding through our submodules before returning an assembled output.

Our `HaikuEnsemble` defines two submodules in `__init__`.

1. `writer` is similar to our last `ReAct` program. We’ve added a new input field, `num_haikus`, specifying how many haikus we want the model to draft. And we’ve changed our output field to return a `list` of strings.
2. `judge` is entirely new. It accepts the `location`, `season`, and `mood` inputs in addition to the candidate `haikus`. It selects the most evocative of the bunch.

When we call this program, our `forward` method runs each module in sequence, then returns a single `dspy.Prediction` object containing our results.

Let's call this module to see which haiku gets selected:

In [5]:
ensemble = HaikuEnsemble(n=3)
result = ensemble(location="Bodega Bay", season="autumn", mood="inspired")

Let's see the candidates:

In [11]:
for c in result.candidates:
    print(c)
    print("-"*10)

Crimson leaves fall soft,
Waves whisper tales of the sea,
Autumn's quiet dance.
----------
Seagulls soar above,
Golden horizon melting,
Inspired by the breeze.
----------
Seaside's gentle hum,
Colors fade into the waves,
Creative spirits.
----------


The model's reasoning:

In [7]:
print(result.reasoning)

The candidates are poetic lines that evoke imagery related to Bodega Bay during autumn, with themes aligned to the mood of being inspired. The second candidate explicitly mentions seagulls, the sea, and a golden horizon, making a direct connection to the seaside and the inspiring nature of the location. The first focuses on autumn leaves and waves, while the third emphasizes the colors fading and creative spirits, which are also evocative. However, the second candidate most vividly captures the seaside environment combined with the season and inspiration, making it the most evocative choice.


And finally, the haiku selected:

In [8]:
print(result.haiku)

Seagulls soar above,
Golden horizon melting,
Inspired by the breeze.


## Using a bigger model as our judge

To make this module a true ensemble, let’s use a different model to grade the work of our haiku writer.

The [`with dspy.context()`](https://dspy.ai/diving-deeper/settings-and-context/) statement allows us to define a new context that sets a new model for the judge call.

We only need to add one line:

In [ ]:
class HaikuEnsemble(dspy.Module):
    def __init__(self, n: int = 3):
        super().__init__()
        self.n = n  
        # Module 1 generates several haikus
        self.writer = dspy.ChainOfThought(
            "location, season, mood, num_haikus: int -> haikus: list[str]",
        )
        # Module 2 picks the most evocative
        self.judge = dspy.ChainOfThought(
            "location, season, mood, candidates: list[str] -> most_evocative_index: int"
        )

    def forward(self, location: str, season: str, mood: str) -> dspy.Prediction:
        candidates = self.writer(
            location=location, season=season, mood=mood, num_haikus=self.n,
        ).haikus
        # Call a much larger model to evaluate our haikus
        with dspy.context(lm=dspy.LM("openai/gpt-5.4")):
            verdict = self.judge( 
                location=location, season=season, mood=mood, candidates=candidates,
            )
        return dspy.Prediction(
            haiku=candidates[verdict.most_evocative_index],
            candidates=candidates,
            reasoning=verdict.reasoning,
        )

## Decompose to isolate, reuse, govern, and optimize

Our haiku task is a small example, but building `HaikuEnsemble` demonstrates how easily we can decompose our programs when necessary. There’s no esoteric chaining API; modules are just Python and the DSPy primitives `Signature`, `Module`, and `LM`.

Reasons to decompose appear as our AI programs grow in complexity and we learn their failure modes. For example, we can use custom modules to:

- **Isolate context:** A wandering investigation step can entertain many candidate subjects, but the final haiku writing call should only see the chosen one. Splitting them keeps each module focused on its own job.
- **Reusable parts across programs:** A well-tuned research submodule isn’t haiku-specific; it could assist any program that needs trusted grounding. Decomposing lets us reuse it across programs.
- **Route easy work to cheaper models:** Lots of small model calls can gather grounding details quickly and cheaply, while a single call to a stronger model handles the final, nuanced composition.
- **Design custom control flow:** We could run the final haiku through a syllable-count check with an NLP library, and call the writer again if the meter is off.
- **Govern independently inspect steps:** Each submodule is its own object, and each call lands in `inspect_history` separately. When the agent does something surprising, we can call one submodule on its own to see exactly what it returned. For haiku writing this is less of a concern, but for high-risk tasks this ability to audit is critical.
- **More easily evaluate and optimize:** Scoring a single haiku in isolation is hard; “good” cuts across too many dimensions to grade cleanly. Picking the best of three is much easier. Decomposing isolates subtasks we can actually score, which enables evaluation and optimization.

See [Modules: composing your own](https://dspy.ai/diving-deeper/modules/) for control-flow patterns (branching, retry loops, parallel calls) and composition beyond this example.

Speaking of evaluation and optimization, it’s time for the next section.